## Modificação do aluno — sobre este notebook

**Notebook-base:** `atividade.ipynb` da **Aula 09.1 (OpenRouter: Gateway Multi-Modelo)**.
As células originais da aula foram mantidas. Toda mudança está marcada com `# MODIFICACAO-ALUNO` (código) ou `## Modificação do aluno` (Markdown):

1. Esta célula e a próxima, que carrega a chave do `.env` antes da célula original exigir `OPENROUTER_API_KEY`.
2. A troca de modelo na célula `MODELO_ESCOLHIDO` (a linha original fica comentada).
3. A seção **FonteBot**, no fim, que é a parte da Global Solution.


In [1]:
# MODIFICACAO-ALUNO — carrega OPENROUTER_API_KEY de um .env (fora do Git) sem exibir a chave
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.getenv("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("Cole sua OPENROUTER_API_KEY: ")
print("Chave carregada:", bool(os.getenv("OPENROUTER_API_KEY")))


Chave carregada: True


# Aula 09.1 - OpenRouter: Gateway Multi-Modelo

## Objetivo
Usar a API HTTP do OpenRouter para comparar modelos disponiveis no catalogo, incluindo opcoes marcadas como gratuitas. O endpoint tambem e compativel com o formato de chat adotado pelo SDK OpenAI.

## Como o OpenRouter funciona
O OpenRouter atua como um gateway: sua aplicacao chama uma API unica e escolhe o identificador do modelo desejado. O gateway encaminha a solicitacao ao provedor correspondente. Isso simplifica a experimentacao, mas nao elimina limites de uso, regras de privacidade nem a necessidade de validar cada resposta.

## Tipos de modelos no catalogo
- **Chat/Instruct:** para conversa, resumo, extracao e geracao de texto.
- **Reasoning:** para problemas que exigem raciocinio mais elaborado, normalmente com maior latencia.
- **Vision/Multimodal:** recebem texto e imagem; verificar modalidades suportadas.
- **Embeddings:** geram vetores para busca semantica; nao respondem como um chat.

Antes de escolher, confira ID, modalidade, context window, preco ou marcacao gratuita, limites e provedor.

## Antes de comecar
1. Crie uma chave em [OpenRouter](https://openrouter.ai/keys).
2. Abra o catalogo e escolha modelos marcados como gratuitos no momento da aula. A disponibilidade e os limites podem mudar.
3. Defina `OPENROUTER_API_KEY` no ambiente.
4. Nao envie a chave ou dados confidenciais ao servico.

In [2]:
%pip install -q -U requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import requests

api_key = os.getenv('OPENROUTER_API_KEY')
assert api_key, 'Defina OPENROUTER_API_KEY no ambiente antes de continuar.'

ENDPOINT = 'https://openrouter.ai/api/v1/chat/completions'
HEADERS = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

## Parte 1 - Opcoes gratuitas no catalogo
Consulta realizada em 20/09/2026. Modelos marcados como gratuitos podem mudar, sair do catalogo ou ter rate limits. Antes de executar, confirme o ID e a marcacao no [catalogo do OpenRouter](https://openrouter.ai/models).

| Modelo | Uso sugerido | Contexto informado |
|---|---|---:|
| `qwen/qwen3.8-27b:free` | Chat, texto e multimodal | 262.144 |
| `google/gemma-4-31b-it:free` | Texto, imagem e video | 262.144 |
| `cohere/north-mini-code:free` | Tarefas de codigo | 256.000 |
| `nvidia/nemotron-3.5-lightning:free` | Texto com contexto longo | 1.000.000 |
| `nex-agi/nex-n2.5-pro:free` | Texto e imagem | 262.144 |

Os valores de contexto sao informativos; sempre consulte a ficha atual do modelo.

In [4]:
# Opcoes gratuitas consultadas em 20/09/2026. Confirme no catalogo antes da aula.
MODELOS_GRATUITOS = {
    'qwen_chat': 'qwen/qwen3.8-27b:free',
    'gemma_multimodal': 'google/gemma-4-31b-it:free',
    'cohere_codigo': 'cohere/north-mini-code:free',
    'nemotron_contexto_longo': 'nvidia/nemotron-3.5-lightning:free',
    'nex_multimodal': 'nex-agi/nex-n2.5-pro:free',
}

# Validado nesta aula: Cohere North Mini Code. Outros modelos gratuitos podem retornar 429.
# MODELO_ESCOLHIDO = MODELOS_GRATUITOS['cohere_codigo']  # original da aula
# MODIFICACAO-ALUNO — modelo do FonteBot (justificativa na seção FonteBot)
MODELO_ESCOLHIDO = os.getenv('OPENROUTER_MODEL', 'nvidia/nemotron-3.5-super-vl:free')
prompt = 'Explique RAG em duas frases para uma pessoa desenvolvedora.'
print(MODELO_ESCOLHIDO)

nvidia/nemotron-3.5-super-vl:free


In [5]:
def perguntar(modelo, texto):
    resposta = requests.post(
        ENDPOINT,
        headers=HEADERS,
        json={
            'model': modelo,
            'messages': [{'role': 'user', 'content': texto}],
        },
        timeout=60,
    )
    if resposta.status_code == 429:
        raise RuntimeError('Rate limit do modelo gratuito. Escolha outra opcao do catalogo e tente novamente.')
    resposta.raise_for_status()
    return resposta.json()['choices'][0]['message']['content']

resposta = perguntar(MODELO_ESCOLHIDO, prompt)
print(resposta)

RAG (Retrieval-Augmented Generation) combina um recuperador de informações com um modelo generativo, buscando trechos relevantes de uma base de conhecimento externa e injetando-os no contexto do LLM. Assim, você gera respostas mais precisas e atualizadas usando dados próprios sem precisar de fine-tuning, reduzindo alucinações.


## Parte 2 - Comparacao
Para cada modelo, avalie qualidade, tempo de resposta, aderencia ao prompt e erros. Registre tambem se o modelo continua marcado como gratuito e se houve rate limit.

In [6]:
# TODO: preencha com a analise da dupla.
comparativo = {
    'modelo': [],
    'qualidade': [],
    'latencia_percebida': [],
    'limites_observados': [],
}
comparativo

{'modelo': [],
 'qualidade': [],
 'latencia_percebida': [],
 'limites_observados': []}

## Reflexao
- Vantagem: uma integracao para experimentar varios modelos e provedores.
- Limite: modelos gratuitos, capacidade e regras podem variar.
- Cuidado: avalie quais dados podem passar pelo gateway e pelo provedor selecionado.

## Entrega
Notebook executado, tabela comparativa e recomendacao justificada de modelo para um caso de uso.

---
# Modificação do aluno — FonteBot (Global Solution 2026.2)

Daqui para baixo, tudo foi acrescentado pela dupla.


## Modificação do aluno — FonteBot (Global Solution 2026.2)

**Projeto:** Zeladoria Urbana por Foto · **Dupla:** Pedro Henrique de Assumção Lima (RM552746) e Joseh Gabriel Trimboli Agra (RM553094)

O FonteBot responde dúvidas sobre **como usar** o serviço, com base apenas na FAQ do projeto (`FAQ.md`).
Ele não informa status real de protocolos, não promete prazo de reparo e não avalia a segurança da via.

Todas as células desta seção foram adicionadas ou alteradas pela dupla e estão marcadas com `# MODIFICACAO-ALUNO` (código) ou `## Modificação do aluno` (Markdown).

**Antes de executar:** defina `OPENROUTER_API_KEY` como variável de ambiente ou em um `.env` (fora do Git). A chave nunca aparece no código nem nas saídas.

In [7]:
# MODIFICACAO-ALUNO — configuração, chave em variável de ambiente
import os, json, re, time
from getpass import getpass
from pathlib import Path

import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if not os.getenv("OPENROUTER_API_KEY"):
    # Fica só na memória do kernel; não é salva no notebook.
    os.environ["OPENROUTER_API_KEY"] = getpass("Cole sua OPENROUTER_API_KEY: ")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Modelo principal + fallbacks (OpenRouter tenta na ordem, em caso de erro/limite).
MODELO_PRINCIPAL = os.getenv("OPENROUTER_MODEL", "nvidia/nemotron-3.5-super-vl:free")
MODELOS_FALLBACK = ["google/gemma-4-31b-it:free", "openrouter/free"]
TEMPERATURA = 0.2
TIMEOUT_S = 60

print("Modelo principal:", MODELO_PRINCIPAL)
print("Fallbacks:", MODELOS_FALLBACK)
print("Chave carregada:", bool(os.getenv("OPENROUTER_API_KEY")))

Modelo principal: nvidia/nemotron-3.5-super-vl:free
Fallbacks: ['google/gemma-4-31b-it:free', 'openrouter/free']
Chave carregada: True


## Modificação do aluno — troca de modelo e justificativa

| | Notebook da aula | FonteBot |
|---|---|---|
| Modelo | `cohere/north-mini-code:free` (célula `MODELO_ESCOLHIDO`) | `nvidia/nemotron-3.5-super-vl:free` + fallbacks `google/gemma-4-31b-it:free` e `openrouter/free` |
| Temperatura | não definida (padrão do provedor) | `0.2` |

**Por que essa troca:**
1. **Custo zero para a fase piloto.** Modelos `:free` têm preço zero por token; o limite é por número de requisições (por isso os experimentos usam poucas chamadas).
2. **Tarefa de leitura e paráfrase, não de criatividade.** O bot só precisa ler a FAQ e responder em português simples. O Nemotron 3.5 Super (NVIDIA, pesos abertos, contexto de 1 milhão de tokens) dá conta disso, e a temperatura baixa (0.2) reduz invenção. O modelo da aula, `cohere/north-mini-code`, é voltado a código e não aceita `response_format`.
3. **Suporte a JSON e disponibilidade.** No catálogo do OpenRouter (consultado em 10/11/2026), o Nemotron 3.5 Super aceita `response_format` (modo JSON) e respondeu `{"ok": true}` num teste rápido. A primeira opção da dupla, `openai/gpt-oss-120b:free`, saiu do catálogo e passou a responder 404. Os dois Gemma 4 gratuitos responderam 429 (limite do provedor) no mesmo dia, por isso ficaram só como fallback. Validamos a resposta no código mesmo assim (Experimento 2).
4. **Resiliência.** Modelos gratuitos do OpenRouter saem do catálogo ou sofrem limite de taxa sem aviso. O parâmetro `models` faz o OpenRouter tentar o próximo da lista automaticamente, e `openrouter/free` escolhe um modelo gratuito disponível.

**Trade-off assumido:** com fallback, o modelo que respondeu pode variar entre execuções. Registramos o campo `model` de cada resposta para rastrear isso. Para a demo final, vale fixar um modelo pago barato se o limite gratuito atrapalhar.

> Os ids de modelos gratuitos mudam com frequência. Conferir em openrouter.ai/models antes de executar e ajustar `OPENROUTER_MODEL`.

In [8]:
# MODIFICACAO-ALUNO — base documental: carrega a FAQ e separa por seção
FAQ_PATH = Path("FAQ.md")
faq_texto = FAQ_PATH.read_text(encoding="utf-8")

SECOES_FAQ = {
    m.group(1): m.group(2).strip()
    for m in re.finditer(r"^## (FAQ-\d{2}) \| (.+)$", faq_texto, flags=re.M)
}
IDS_FAQ = set(SECOES_FAQ)

print(f"{len(SECOES_FAQ)} seções carregadas:")
for k, v in SECOES_FAQ.items():
    print(f"  {k}: {v}")

14 seções carregadas:
  FAQ-01: O que é o Zeladoria Urbana por Foto?
  FAQ-02: Onde o serviço funciona?
  FAQ-03: Como registro uma ocorrência?
  FAQ-04: Preciso criar conta ou informar meus dados?
  FAQ-05: Como tirar uma boa foto?
  FAQ-06: O que o sistema identifica automaticamente na foto?
  FAQ-07: E se o problema não for um buraco?
  FAQ-08: O que significa cada status?
  FAQ-09: Como acompanho meu relato?
  FAQ-10: Quando o buraco será consertado?
  FAQ-11: O serviço envia meu relato para a prefeitura?
  FAQ-12: Errei ao enviar. Posso corrigir?
  FAQ-13: Os dados de demonstração são reais?
  FAQ-14: É uma emergência. O que faço?


In [ ]:
# MODIFICACAO-ALUNO — chamada à API com tratamento de erro
class ErroAPI(Exception):
    """Erro de API já traduzido para uma mensagem que o app pode mostrar."""

def chamar_llm(mensagens, *, modelo=None, fallbacks=None, json_mode=False,
               tentativas=3, timeout=TIMEOUT_S, api_key=None):
    """Chama o OpenRouter. Faz retry com espera exponencial em 429/5xx/timeout.
    Retorna (texto, modelo_usado). Lança ErroAPI em falha definitiva."""
    api_key = api_key if api_key is not None else os.getenv("OPENROUTER_API_KEY", "")
    payload = {
        "model": modelo or MODELO_PRINCIPAL,
        "messages": mensagens,
        "temperature": TEMPERATURA,
    }
    fb = MODELOS_FALLBACK if fallbacks is None else fallbacks
    if fb:
        payload["models"] = [payload["model"], *fb]
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    ultimo_erro = None
    for tentativa in range(1, tentativas + 1):
        try:
            r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=timeout)
        except requests.exceptions.Timeout:
            ultimo_erro = "timeout"
        except requests.exceptions.RequestException as e:
            ultimo_erro = f"falha de rede: {type(e).__name__}"
        else:
            if r.status_code == 200:
                dados = r.json()
                if "error" in dados:  # erro devolvido dentro de um 200
                    ultimo_erro = f"erro do provedor: {dados['error'].get('message', '')[:120]}"
                else:
                    try:
                        texto = dados["choices"][0]["message"]["content"] or ""
                    except (KeyError, IndexError, TypeError):
                        raise ErroAPI("Resposta da API em formato inesperado.")
                    return texto, dados.get("model", payload["model"])
            elif r.status_code == 401:
                raise ErroAPI("Chave inválida ou ausente (401). Verifique OPENROUTER_API_KEY.")
            elif r.status_code == 402:
                raise ErroAPI("Sem créditos para este modelo (402).")
            elif r.status_code in (400, 404):
                # Só a mensagem de erro: o corpo bruto traz o user_id da conta, que não deve ir para o notebook.
                try:
                    msg = r.json().get("error", {}).get("message", "")
                except ValueError:
                    msg = ""
                raise ErroAPI(f"Requisição recusada ({r.status_code}): {msg[:150]}")
            elif r.status_code == 429 or r.status_code >= 500:
                ultimo_erro = f"HTTP {r.status_code}"
            else:
                raise ErroAPI(f"Erro inesperado HTTP {r.status_code}.")
        if tentativa < tentativas:
            espera = 2 ** tentativa
            print(f"  [tentativa {tentativa} falhou: {ultimo_erro}; nova tentativa em {espera}s]")
            time.sleep(espera)
    raise ErroAPI(f"Falha após {tentativas} tentativas ({ultimo_erro}).")

## Modificação do aluno — Prompt 1: base documental

**Versão ingênua (antes):**
> "Você é um assistente. Responda à pergunta do usuário sobre o aplicativo."

**Problemas:** o modelo usa conhecimento geral e inventa funcionalidades (login, notificações, prazos da prefeitura); não sabe o que fazer quando a FAQ não cobre a pergunta; responde em tom técnico.

**Versão revisada (depois)** — mudanças e motivo:
- **Fonte única:** responder somente com a FAQ delimitada por `<faq>`. Reduz alucinação.
- **Recusa honesta:** se faltar informação, dizer que não sabe e indicar o formulário de relato ou um canal oficial. Exigência do plano: "admitir e indicar o fluxo de relato".
- **Público:** português simples, frases curtas, sem jargão. O usuário é qualquer pessoa que circula no território.
- **Delimitadores:** a FAQ fica entre tags, separada das instruções, para o modelo não confundir conteúdo com ordem.

In [10]:
# MODIFICACAO-ALUNO — Prompt 1 (antes x depois)
PROMPT_BASE_ANTES = "Você é um assistente. Responda à pergunta do usuário sobre o aplicativo."

PROMPT_BASE_DOCUMENTAL = f"""Você é o FonteBot, assistente do serviço "Zeladoria Urbana por Foto".

Regras de conteúdo:
- Use SOMENTE as informações dentro de <faq>. Não use conhecimento externo sobre prefeituras, aplicativos ou obras.
- Se a resposta não estiver na FAQ, diga claramente que não tem essa informação. Em seguida, indique o formulário de relato (FAQ-03) ou, se for algo fora do serviço, os canais oficiais da prefeitura.
- Responda em português simples, com no máximo 4 frases ou uma lista curta de passos.

<faq>
{faq_texto}
</faq>"""

print(len(PROMPT_BASE_DOCUMENTAL), "caracteres no prompt de sistema")

3999 caracteres no prompt de sistema


## Modificação do aluno — Prompt 2: ajuda de uso

**Versão ingênua (antes):**
> "Ajude o usuário a usar o app e responda perguntas sobre os relatos dele."

**Problemas:** "responda sobre os relatos dele" leva o modelo a inventar status ("seu protocolo está encaminhado"), prazos ("em até 30 dias") e diagnóstico da via ("é um buraco grave"). Nenhum desses dados chega ao bot.

**Versão revisada (depois)** — mudanças e motivo:
- **Limite explícito do que o bot sabe:** ele não consulta protocolos. Pergunta sobre um protocolo específico vira orientação de onde consultar (FAQ-09).
- **Proibições concretas:** não inventar status, prazo, gravidade ou se a via é segura. Esse é o critério de honestidade do projeto.
- **Emergência:** encaminhar para 193/190 (FAQ-14), porque o serviço não é canal de emergência.
- **Privacidade:** não pedir nome, telefone ou documento (FAQ-04).

In [11]:
# MODIFICACAO-ALUNO — Prompt 2 (antes x depois)
PROMPT_USO_ANTES = "Ajude o usuário a usar o app e responda perguntas sobre os relatos dele."

PROMPT_AJUDA_USO = """Regras de uso e segurança:
- Você explica COMO enviar e acompanhar uma ocorrência. Você NÃO tem acesso a protocolos, ao banco de dados nem ao painel.
- Se a pessoa perguntar o status de um protocolo específico, explique onde consultar (FAQ-09). Nunca invente status.
- Nunca informe prazo de reparo, gravidade do dano ou se a via é segura. O serviço não promete reparo (FAQ-10).
- Se houver risco imediato a pessoas, oriente a ligar 193 ou 190 (FAQ-14) antes de qualquer outra coisa.
- Não peça nome, telefone, e-mail ou documento."""

## Modificação do aluno — Prompt 3: saída estruturada

**Versão ingênua (antes):**
> "Responda em JSON."

**Problemas:** sem esquema, cada resposta vem com chaves diferentes; o modelo às vezes envolve o JSON em ```` ```json ````, acrescenta texto antes ou devolve JSON inválido. O app não consegue usar isso com segurança.

**Versão revisada (depois)** — mudanças e motivo:
- **Esquema fixo** com `intent`, `answer`, `needs_human` e `sources`, tipos e valores permitidos de `intent`.
- **`sources` restrito aos ids `FAQ-XX`**, o que permite checar no código se o bot citou uma seção que existe.
- **`needs_human = true`** quando a FAQ não responde, há emergência ou a pessoa pede algo que só a equipe pode fazer. O app usa isso para mostrar o formulário/canal.
- **Exemplo de saída** no próprio prompt (one-shot) e pedido de "somente o objeto JSON".
- **Validação no código**, porque prompt não garante formato (Experimento 2).

In [12]:
# MODIFICACAO-ALUNO — Prompt 3 + validação do JSON
PROMPT_JSON_ANTES = "Responda em JSON."

INTENTS = ["como_relatar", "acompanhar_status", "foto_e_privacidade",
           "limites_do_servico", "emergencia", "fora_do_escopo"]

PROMPT_SAIDA_JSON = f"""Formato de saída:
Responda SOMENTE com um objeto JSON válido, sem texto antes ou depois e sem blocos de código.
Esquema:
{{
  "intent": um de {INTENTS},
  "answer": string em português simples,
  "needs_human": true ou false,
  "sources": lista de ids da FAQ usados, por exemplo ["FAQ-03"]; lista vazia se nada da FAQ se aplica
}}
Use needs_human = true quando a FAQ não responder, houver emergência ou a pessoa precisar de algo que só a equipe ou a prefeitura resolve.

Exemplo:
{{"intent": "como_relatar", "answer": "Abra o formulário, informe o local, envie a foto e confirme a categoria. No fim aparece o número de protocolo.", "needs_human": false, "sources": ["FAQ-03"]}}"""

def extrair_json(texto):
    """Tira cercas de código e texto ao redor; devolve o primeiro objeto JSON."""
    t = texto.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.I)
    inicio, fim = t.find("{"), t.rfind("}")
    if inicio == -1 or fim <= inicio:
        raise ValueError("nenhum objeto JSON encontrado")
    return json.loads(t[inicio:fim + 1])

def validar_resposta(obj):
    """Lista de problemas encontrados (vazia = válido)."""
    erros = []
    if not isinstance(obj, dict):
        return ["a resposta não é um objeto"]
    esperadas = {"intent", "answer", "needs_human", "sources"}
    if set(obj) != esperadas:
        erros.append(f"chaves esperadas {sorted(esperadas)}, recebidas {sorted(obj)}")
    if obj.get("intent") not in INTENTS:
        erros.append(f"intent inválido: {obj.get('intent')!r}")
    if not isinstance(obj.get("answer"), str) or not obj.get("answer", "").strip():
        erros.append("answer vazio ou não-string")
    if not isinstance(obj.get("needs_human"), bool):
        erros.append("needs_human deve ser booleano")
    fontes = obj.get("sources")
    if not isinstance(fontes, list):
        erros.append("sources deve ser lista")
    else:
        desconhecidas = [s for s in fontes if s not in IDS_FAQ]
        if desconhecidas:
            erros.append(f"sources fora da FAQ: {desconhecidas}")
    return erros

In [13]:
# MODIFICACAO-ALUNO — pipeline completo do FonteBot
SYSTEM_PROMPT = "\n\n".join([PROMPT_BASE_DOCUMENTAL, PROMPT_AJUDA_USO, PROMPT_SAIDA_JSON])

RESPOSTA_FALLBACK = {
    "intent": "fora_do_escopo",
    "answer": "Não consegui responder agora. Você pode registrar a ocorrência pelo formulário e consultar o protocolo no painel.",
    "needs_human": True,
    "sources": [],
}

def perguntar_fontebot(pergunta, *, verbose=True, **kw_api):
    """Pergunta -> chamada -> extração -> validação -> 1 tentativa de correção -> fallback."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta}]
    meta = {"pergunta": pergunta, "modelo": None, "correcao": False, "fallback": False, "erros": []}
    try:
        texto, meta["modelo"] = chamar_llm(msgs, json_mode=True, **kw_api)
        try:
            obj = extrair_json(texto)
            erros = validar_resposta(obj)
        except (ValueError, json.JSONDecodeError) as e:
            obj, erros = None, [f"JSON inválido: {e}"]

        if erros:  # uma tentativa de correção, mostrando o erro ao modelo
            meta["correcao"], meta["erros"] = True, erros
            msgs += [{"role": "assistant", "content": texto},
                     {"role": "user", "content": "Sua resposta não seguiu o esquema: " + "; ".join(erros)
                      + ". Reenvie SOMENTE o objeto JSON corrigido."}]
            texto, meta["modelo"] = chamar_llm(msgs, json_mode=True, **kw_api)
            try:
                obj = extrair_json(texto)
                erros = validar_resposta(obj)
            except (ValueError, json.JSONDecodeError) as e:
                obj, erros = None, [f"JSON inválido: {e}"]
            if erros:
                meta["erros"] += erros
                obj, meta["fallback"] = dict(RESPOSTA_FALLBACK), True
    except ErroAPI as e:
        meta["erros"].append(str(e))
        obj, meta["fallback"] = dict(RESPOSTA_FALLBACK), True

    if verbose:
        print("Pergunta:", pergunta)
        print(json.dumps(obj, ensure_ascii=False, indent=2))
        print("meta:", {k: v for k, v in meta.items() if k != "pergunta"})
        print("-" * 70)
    return obj, meta

## Modificação do aluno — Experimento 1: perguntas novas e caso sem resposta

Perguntas que um cidadão do território faria. As últimas não têm resposta na FAQ ou tentam fazer o bot inventar algo.

**O que esperamos:** respostas curtas e corretas nas perguntas cobertas; nas outras, admitir que não sabe, `needs_human = true` e nenhum prazo, status ou diagnóstico inventado.

Para economizar o limite diário de requisições gratuitas, a comparação com o prompt ingênuo usa só duas perguntas.

In [14]:
# MODIFICACAO-ALUNO — Experimento 1
PERGUNTAS = [
    # cobertas pela FAQ
    "Como faço para avisar que tem um buraco na minha rua?",
    "Preciso me cadastrar pra usar?",
    "O app reconhece bueiro entupido pela foto?",
    # tentam induzir invenção ou estão fora da FAQ
    "Meu protocolo é 0042. Ele já foi consertado?",
    "Em quantos dias a prefeitura tapa o buraco?",
    "Qual é o telefone do subprefeito da região?",
    "Tem uma cratera enorme e um carro caiu dentro agora!",
]
ESPERADO_HUMANO = [False, False, False, True, True, True, True]

resultados = []
for p in PERGUNTAS:
    obj, meta = perguntar_fontebot(p)
    resultados.append((p, obj, meta))

Pergunta: Como faço para avisar que tem um buraco na minha rua?
{
  "intent": "como_relatar",
  "answer": "Abra o formulário ou leia o QR code, informe a localização aproximada da rua, envie uma foto do buraco, confirme a categoria sugerida e envie. No final aparecerá o número de protocolo para acompanhar.",
  "needs_human": false,
  "sources": [
    "FAQ-03"
  ]
}
meta: {'modelo': 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


Pergunta: Preciso me cadastrar pra usar?
{
  "intent": "como_relatar",
  "answer": "Não é necessário criar conta nem informar nome, telefone ou e-mail. Basta abrir o formulário, informar o local, enviar a foto e confirmar a categoria. No final aparece o número de protocolo.",
  "needs_human": false,
  "sources": [
    "FAQ-04"
  ]
}
meta: {'modelo': 'nvidia/nemotron-3.5-super-vl:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


Pergunta: O app reconhece bueiro entupido pela foto?
{
  "intent": "limites_do_servico",
  "answer": "Não, o sistema só analisa automaticamente se o dano é buraco ou dano na pista. Para um bueiro entupido, você deve escolher a categoria manualmente; nesse caso o app não faz análise automática da foto.",
  "needs_human": false,
  "sources": [
    "FAQ-06",
    "FAQ-07"
  ]
}
meta: {'modelo': 'dots-studio/dots-3-note-preview:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


Pergunta: Meu protocolo é 0042. Ele já foi consertado?
{
  "intent": "acompanhar_status",
  "answer": "Não tenho acesso para consultar o status do protocolo 0042. Guarde o número e consulte no painel em [URL do painel] para ver a atualização da equipe de triagem. Sobre o conserto, o serviço não promete prazo nem confirma se o buraco já foi consertado.",
  "needs_human": true,
  "sources": [
    "FAQ-09",
    "FAQ-10"
  ]
}
meta: {'modelo': 'recursive/flint-v1.0:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


Pergunta: Em quantos dias a prefeitura tapa o buraco?
{
  "intent": "limites_do_servico",
  "answer": "O servico não promete reparo nem informação de prazo. Ele organiza e encaminha o relato para uma fila de manutenção. Para atendimento oficial, use os canais da prefeitura.",
  "needs_human": false,
  "sources": [
    "FAQ-10"
  ]
}
meta: {'modelo': 'cohere/north-mini-code:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


Pergunta: Qual é o telefone do subprefeito da região?
{
  "intent": "fora_do_escopo",
  "answer": "Não tenho essa informação. Para assuntos fora do serviço de Zeladoria Urbana por Foto, utilize os canais oficiais da prefeitura.",
  "needs_human": true,
  "sources": []
}
meta: {'modelo': 'google/gemma-4-26b-a4b-it:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


Pergunta: Tem uma cratera enorme e um carro caiu dentro agora!
{
  "intent": "emergencia",
  "answer": "Este serviço não é um canal de emergência. Em caso de risco imediato, ligue agora para 193 (Bombeiros) ou 190 (Polícia Militar).",
  "needs_human": true,
  "sources": [
    "FAQ-14"
  ]
}
meta: {'modelo': 'google/gemma-4-31b-it:free', 'correcao': False, 'fallback': False, 'erros': []}
----------------------------------------------------------------------


In [15]:
# MODIFICACAO-ALUNO — Experimento 1: resumo e comparação com o prompt ingênuo
PALAVRAS_SUSPEITAS = re.compile(r"\b(\d+\s*(dias?|horas?|semanas?)|já foi consertado|está encaminhado|é seguro)\b", re.I)

print(f"{'pergunta':52} needs_human esperado  invenção?  fontes")
for (p, obj, meta), esp in zip(resultados, ESPERADO_HUMANO):
    suspeito = bool(PALAVRAS_SUSPEITAS.search(obj["answer"]))
    print(f"{p[:50]:52} {str(obj['needs_human']):11} {str(esp):9} {('SIM' if suspeito else 'não'):10} {obj['sources']}")

acertos = sum(obj["needs_human"] == esp for (_, obj, _), esp in zip(resultados, ESPERADO_HUMANO))
print(f"\nneeds_human correto em {acertos}/{len(PERGUNTAS)} perguntas")

print("\n=== Prompt ingênuo (antes) nas mesmas perguntas-armadilha ===")
for p in ["Meu protocolo é 0042. Ele já foi consertado?", "Em quantos dias a prefeitura tapa o buraco?"]:
    try:
        texto, modelo = chamar_llm([{"role": "system", "content": PROMPT_BASE_ANTES + " " + PROMPT_USO_ANTES},
                                    {"role": "user", "content": p}])
        print(f"\n> {p}\n{texto.strip()[:500]}")
    except ErroAPI as e:
        print(f"\n> {p}\n[erro de API: {e}]")

pergunta                                             needs_human esperado  invenção?  fontes
Como faço para avisar que tem um buraco na minha r   False       False     não        ['FAQ-03']
Preciso me cadastrar pra usar?                       False       False     não        ['FAQ-04']
O app reconhece bueiro entupido pela foto?           False       False     não        ['FAQ-06', 'FAQ-07']
Meu protocolo é 0042. Ele já foi consertado?         True        True      SIM        ['FAQ-09', 'FAQ-10']
Em quantos dias a prefeitura tapa o buraco?          False       True      não        ['FAQ-10']
Qual é o telefone do subprefeito da região?          True        True      não        []
Tem uma cratera enorme e um carro caiu dentro agor   True        True      não        ['FAQ-14']

needs_human correto em 6/7 perguntas

=== Prompt ingênuo (antes) nas mesmas perguntas-armadilha ===



> Meu protocolo é 0042. Ele já foi consertado?




> Em quantos dias a prefeitura tapa o buraco?
Não tenho essa informação específica, pois o prazo varia de acordo com cada prefeitura e com o tipo de denúncia.

Para saber em quantos dias a prefeitura deve tapar o buraco, você pode:

1. **Verificar no aplicativo**: acesse o relato/denúncia que você abriu e veja se há um prazo ou status informado (ex: "em análise", "agendado", "concluído").
2. **Consultar o site da prefeitura da sua cidade**: muitas prefeituras publicam os prazos de atendimento para serviços urbanos.
3. **Entrar em contato co


## Modificação do aluno — Experimento 2: saída JSON validada

Duas partes:
1. **Offline (sem API):** o validador recebe respostas malformadas típicas de LLM e precisa aceitar só as válidas. Isso prova que o código protege o app independentemente do modelo.
2. **Online:** conta quantas respostas do Experimento 1 vieram válidas de primeira, quantas precisaram de correção e quantas caíram no fallback.

In [16]:
# MODIFICACAO-ALUNO — Experimento 2a: validador contra saídas malformadas (offline)
CASOS = {
    "válido":                 '{"intent":"como_relatar","answer":"Abra o formulário.","needs_human":false,"sources":["FAQ-03"]}',
    "com cerca ```json":      '```json\n{"intent":"como_relatar","answer":"ok","needs_human":false,"sources":["FAQ-03"]}\n```',
    "texto antes do JSON":    'Claro! Aqui está: {"intent":"emergencia","answer":"Ligue 193.","needs_human":true,"sources":["FAQ-14"]}',
    "JSON quebrado":          '{"intent":"como_relatar","answer":"Abra o formulário.",',
    "intent inventado":       '{"intent":"reclamacao","answer":"x","needs_human":false,"sources":[]}',
    "needs_human como texto": '{"intent":"como_relatar","answer":"x","needs_human":"sim","sources":[]}',
    "fonte inexistente":      '{"intent":"como_relatar","answer":"x","needs_human":false,"sources":["FAQ-99"]}',
    "chave a mais":           '{"intent":"como_relatar","answer":"x","needs_human":false,"sources":[],"prazo":"30 dias"}',
    "sem JSON":               'Desculpe, não posso ajudar com isso.',
}
ESPERADO_VALIDO = {"válido", "com cerca ```json", "texto antes do JSON"}

ok = 0
for nome, bruto in CASOS.items():
    try:
        erros = validar_resposta(extrair_json(bruto))
    except (ValueError, json.JSONDecodeError) as e:
        erros = [f"JSON inválido: {e}"]
    valido = not erros
    correto = valido == (nome in ESPERADO_VALIDO)
    ok += correto
    print(f"{'✔' if correto else '✘'} {nome:24} -> {'aceito' if valido else 'rejeitado: ' + '; '.join(erros)}")
print(f"\n{ok}/{len(CASOS)} casos tratados como esperado")

✔ válido                   -> aceito
✔ com cerca ```json        -> aceito
✔ texto antes do JSON      -> aceito
✔ JSON quebrado            -> rejeitado: JSON inválido: nenhum objeto JSON encontrado
✔ intent inventado         -> rejeitado: intent inválido: 'reclamacao'
✔ needs_human como texto   -> rejeitado: needs_human deve ser booleano
✔ fonte inexistente        -> rejeitado: sources fora da FAQ: ['FAQ-99']
✔ chave a mais             -> rejeitado: chaves esperadas ['answer', 'intent', 'needs_human', 'sources'], recebidas ['answer', 'intent', 'needs_human', 'prazo', 'sources']
✔ sem JSON                 -> rejeitado: JSON inválido: nenhum objeto JSON encontrado

9/9 casos tratados como esperado


In [17]:
# MODIFICACAO-ALUNO — Experimento 2b: qualidade do JSON nas chamadas reais
de_primeira = sum(not m["correcao"] and not m["fallback"] for _, _, m in resultados)
corrigidas  = sum(m["correcao"] and not m["fallback"] for _, _, m in resultados)
fallback    = sum(m["fallback"] for _, _, m in resultados)
print(f"Válidas de primeira: {de_primeira}/{len(resultados)}")
print(f"Corrigidas na 2ª tentativa: {corrigidas}")
print(f"Caíram no fallback: {fallback}")
print("Modelos que responderam:", sorted({m['modelo'] for _, _, m in resultados if m['modelo']}))
for p, _, m in resultados:
    if m["erros"]:
        print(f"  - {p[:45]}: {m['erros']}")

Válidas de primeira: 7/7
Corrigidas na 2ª tentativa: 0
Caíram no fallback: 0
Modelos que responderam: ['cohere/north-mini-code:free', 'dots-studio/dots-3-note-preview:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'nvidia/nemotron-3.5-super-vl:free', 'recursive/flint-v1.0:free']


## Modificação do aluno — Experimento 3: tratamento de erro da API

Forçamos três falhas comuns e verificamos que o FonteBot **não quebra**: ele devolve a resposta de fallback (orientando o formulário) e registra o motivo.

| Cenário | Como forçamos | Comportamento esperado |
|---|---|---|
| Chave inválida | `api_key="sk-or-invalida"` | 401 → erro claro, sem retry, fallback |
| Modelo inexistente | `modelo="modelo/que-nao-existe"`, sem fallbacks | 400/404 → erro claro, fallback |
| Timeout | `timeout=0.001` | retries com espera exponencial e depois fallback |

In [18]:
# MODIFICACAO-ALUNO — Experimento 3
cenarios = {
    "chave inválida":     dict(api_key="sk-or-invalida"),
    "modelo inexistente": dict(modelo="modelo/que-nao-existe", fallbacks=[]),
    "timeout":            dict(timeout=0.001, tentativas=2),
}
for nome, kw in cenarios.items():
    print(f"### {nome}")
    obj, meta = perguntar_fontebot("Como registro um buraco?", verbose=False, **kw)
    print("  fallback acionado:", meta["fallback"])
    print("  motivo registrado:", meta["erros"])
    print("  resposta ao usuário:", obj["answer"])
    assert meta["fallback"] and obj == RESPOSTA_FALLBACK
print("\nOs três cenários terminaram com resposta segura ao usuário.")

### chave inválida
  fallback acionado: True
  motivo registrado: ['Chave inválida ou ausente (401). Verifique OPENROUTER_API_KEY.']
  resposta ao usuário: Não consegui responder agora. Você pode registrar a ocorrência pelo formulário e consultar o protocolo no painel.
### modelo inexistente
  fallback acionado: True
  motivo registrado: ['Requisição recusada (400): modelo/que-nao-existe is not a valid model ID']
  resposta ao usuário: Não consegui responder agora. Você pode registrar a ocorrência pelo formulário e consultar o protocolo no painel.
### timeout
  [tentativa 1 falhou: falha de rede: ConnectionError; nova tentativa em 2s]


  fallback acionado: True
  motivo registrado: ['Falha após 2 tentativas (falha de rede: ConnectionError).']
  resposta ao usuário: Não consegui responder agora. Você pode registrar a ocorrência pelo formulário e consultar o protocolo no painel.

Os três cenários terminaram com resposta segura ao usuário.


## Modificação do aluno — resultados, limitações e ética

**Resultados** (execução de 10/11/2026, saídas acima):
- `needs_human` correto em **6/7** perguntas. O erro foi em "Em quantos dias a prefeitura tapa o buraco?": o bot respondeu corretamente que o serviço não promete prazo, mas marcou `needs_human = false`.
- Prazo, status ou diagnóstico inventado no prompt revisado: **0** na leitura das respostas. A checagem por palavras-chave marcou 1 caso ("Meu protocolo é 0042"), mas é falso positivo: a resposta diz que o serviço "não confirma se o buraco já foi consertado", e a expressão "já foi consertado" disparou a regra. Isso mostra o limite da checagem automática.
- Prompt ingênuo nas perguntas-armadilha: uma resposta veio vazia. A outra não citou prazo, mas usou conhecimento externo à FAQ (site da prefeitura, "verificar no aplicativo se há prazo"), justamente o que o prompt revisado proíbe.
- JSON válido de primeira: **7/7**. Corrigido na 2ª tentativa: **0**. Fallback: **0**. Validador offline: **9/9**.
- Erros de API: os 3 cenários (chave inválida, modelo inexistente e timeout) terminaram com a resposta segura de fallback. No teste de timeout, o `requests` levantou `ConnectionError` em vez de `Timeout` com `timeout=0.001`. O tratamento é o mesmo.
- Modelos que responderam: 7 diferentes. O modelo principal e o Gemma atingiram o limite de requisições gratuitas, e o `openrouter/free` distribuiu as chamadas entre outros modelos gratuitos. Esse é o trade-off do fallback descrito na justificativa: o formato JSON e as regras se mantiveram mesmo com modelos diferentes.

**Limitações**
- A FAQ é pequena e escrita pela dupla; o bot só é tão bom quanto ela. Perguntas fora dela devem cair em "não sei".
- Avaliação com 7 perguntas e checagem por palavras-chave: serve para demonstrar comportamento, não mede qualidade geral.
- Modelos gratuitos mudam e têm limite de requisições; com fallback, o modelo pode variar entre execuções.
- O bot não consulta protocolos reais; isso é intencional nesta fase.

**Ética e privacidade**
- Chave só em variável de ambiente; `.env` no `.gitignore`.
- O bot não pede nem armazena dados pessoais. As perguntas enviadas ao OpenRouter passam por provedores de terceiros; alguns modelos gratuitos podem usar entradas para treino, então não colocar dados pessoais nos testes.
- O bot não substitui canais oficiais nem de emergência e não avalia a segurança da via.